In [1]:
from src import preprocess
from src import run_cicero

In [2]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread

In [3]:
counts = mmread("./make_cicero_cds_R_Objs/full_matrix.mtx").T.tocsr()
obs_names = pd.read_csv("./make_cicero_cds_R_Objs/full_meta.csv", header = 0, index_col = 0)
var_names = pd.read_csv("./make_cicero_cds_R_Objs/full_matrix_rownames.csv", header = 0, index_col = 1)
var_names.drop("Unnamed: 0", axis = 1, inplace = True)
# obs_names.drop("Unnamed: 0", axis = 1, inplace = True)

In [4]:
adata = sc.AnnData(counts, obs = obs_names, var = var_names)

In [5]:
adata

AnnData object with n_obs × n_vars = 37584 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group'

In [6]:
adata = preprocess.preprocess_cicero(adata)

preprocess.py: 2025-04-11 18:30:55 WARNING  Counts layers not found. Coppying X to counts
preprocess.py: 2025-04-11 18:30:56 INFO     Estimaing Size Factors and Normalizing Data
preprocess.py: 2025-04-11 18:31:03 INFO     Finished Size Factors and Normallization storing normalized sparse matrix into 'data'
preprocess.py: 2025-04-11 18:31:03 INFO     Running TF-IDF
preprocess.py: 2025-04-11 18:31:07 INFO     Finsihed TF-IDF
preprocess.py: 2025-04-11 18:31:07 INFO     Running TruncatedSVD
preprocess.py: 2025-04-11 18:35:41 INFO     Finsihed TruncatedSVD
preprocess.py: 2025-04-11 18:35:41 INFO     Running UMAP
preprocess.py: 2025-04-11 18:35:42 INFO     Finsihed UMAP


In [7]:
cicero_adata = preprocess.make_cicero_adata(adata)

preprocess.py: 2025-04-11 18:35:42 INFO     Using adata OBSM X_umap to agregate cells
preprocess.py: 2025-04-11 18:35:42 INFO     Calculating overlap
preprocess.py: 2025-04-11 18:35:42 INFO     Generating Pseudobulk cicero observations with seed: 0
100%|█████████▉| 4999/5000 [00:05<00:00, 983.51it/s] 
preprocess.py: 2025-04-11 18:35:48 INFO     Reached Maximum itterations in pseudobulk observation generation. Consider increasing 'max_itterations'
preprocess.py: 2025-04-11 18:35:48 INFO     Found 4824 good choices
preprocess.py: 2025-04-11 18:35:48 INFO     Finished calculating overlap
preprocess.py: 2025-04-11 18:35:48 INFO     Aggregating Cells
preprocess.py: 2025-04-11 18:35:50 INFO     Finished Aggregating Cells


In [8]:
cicero_adata

AnnData object with n_obs × n_vars = 4824 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group'

In [9]:
# Preprocess the index strings, assuming format "chr-start-end" (e.g., "chr-0-1923") 
# Also sorted from lower to higher
temp_df = pd.DataFrame([x.split("-") for x in cicero_adata.var.index])
cicero_adata.var["Chromosome"] = temp_df.iloc[:, 0].values
cicero_adata.var["Start"] = temp_df.iloc[:, 1].astype(int).values
cicero_adata.var["End"] = temp_df.iloc[:, 2].astype(int).values

In [10]:
#Usually sorted, if not sort
chromosomes_ordered = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8',
       'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
       'chr16', 'chr17', 'chr18', 'chr19', 'chrX', 'chrY', 'GL456211.1',
       'GL456216.1', 'GL456233.1', 'JH584295.1', 'JH584304.1']
temp_vars = []
for chromosome in chromosomes_ordered:
    chromosome_var = cicero_adata.var[cicero_adata.var["Chromosome"] == chromosome]
    chromosome_var.sort_values("Start", ascending = False) #lower to higher
    temp_vars.append(chromosome_var)
cicero_adata.var = pd.concat(temp_vars, axis = 0)

In [11]:
cicero_adata.var

,Chromosome,Start,End
x,,,
chr1-3119740-3120239,chr1,3119740,3120239
chr1-3121226-3121725,chr1,3121226,3121725
chr1-3155081-3155580,chr1,3155081,3155580
chr1-3203852-3204351,chr1,3203852,3204351
chr1-3210121-3210620,chr1,3210121,3210620
...,...,...,...
JH584304.1-67266-67765,JH584304.1,67266,67765
JH584304.1-67886-68385,JH584304.1,67886,68385
JH584304.1-68747-69246,JH584304.1,68747,69246


In [12]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")
"""
probably from inverse_covriance (skggm)
from: Starting distance_parameter_estimation
/home/twoo/miniconda3/envs/rapids_SC/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
"""

"\nprobably from inverse_covriance (skggm)\nfrom: Starting distance_parameter_estimation\n/home/twoo/miniconda3/envs/rapids_SC/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.\n  warnings.warn(\n"

In [13]:
cons = run_cicero.run_cicero(cicero_adata)

run_cicero.py: 2025-04-11 18:35:51 INFO     Starting multiprocessing pool for estimate_distance_parameter_parallel with 96 processes
run_cicero.py: 2025-04-11 18:36:05 INFO     Generating Windows
run_cicero.py: 2025-04-11 18:36:06 INFO     Starting distance_parameter_estimation
run_cicero.py: 2025-04-11 18:36:53 INFO     Finished distance_parameter_estimation
run_cicero.py: 2025-04-11 18:36:53 INFO     Starting generate_cicero_models
run_cicero.py: 2025-04-11 18:36:56 INFO     Starting Cicero with 96 processes
run_cicero.py: 2025-04-11 18:37:08 INFO     Finished generate_cicero_models
run_cicero.py: 2025-04-11 18:37:09 INFO     Starting assemble_connections
run_cicero.py: 2025-04-11 18:37:32 INFO     Finished assemble_connections
